# Interpretability From Scratch

**Build a tiny transformer and interpret it — no libraries, just PyTorch**

---

Before we use any interpretability libraries, we're going to build a tiny transformer, train it on a simple task, and then poke around inside it with our own code. By the end you'll understand exactly what TransformerLens does under the hood, because you'll have built a mini version yourself.

The philosophy here is Karpathy-style: **if you can't build it from scratch, you don't understand it.** We'll implement:

1. A 2-layer, 2-head transformer from scratch in raw PyTorch
2. Train it on a simple sequence-copying task
3. Inspect the residual stream
4. Visualize attention patterns
5. Build a logit lens
6. Implement activation patching
7. Analyze OV circuits

No TransformerLens. No fancy libraries. Just `torch`, `numpy`, and `matplotlib`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")
print(f"Using device: {device}")

## Part 1: A Tiny Transformer

We'll build a transformer with these specs:

| Parameter | Value |
|---|---|
| Layers | 2 |
| Heads per layer | 2 |
| d_model | 64 |
| d_head | 32 |
| Vocab size | 20 |
| Context length | 16 |

This is deliberately tiny — small enough to fit every weight in your head. The architecture follows GPT-2 style (pre-norm with LayerNorm), and we store **every intermediate activation** so we can inspect the model's internals.

Key design choice: we keep W_Q, W_K, W_V, W_O as **separate per-head parameters** rather than stacking them into big matrices. This makes it easy to inspect individual heads later.

In [ ]:
# === Model hyperparameters ===
N_LAYERS = 2
N_HEADS = 2
D_MODEL = 64
D_HEAD = 32
VOCAB_SIZE = 20
CTX_LEN = 16


class Attention(nn.Module):
    """Multi-head attention with separate per-head weight matrices.
    
    We store W_Q, W_K, W_V, W_O individually per head so we can
    inspect them later for circuit analysis.
    """
    def __init__(self, d_model, n_heads, d_head, ctx_len):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        
        # Per-head projection matrices: easy to inspect individually
        self.W_Q = nn.ParameterList([nn.Parameter(torch.randn(d_model, d_head) * 0.02) for _ in range(n_heads)])
        self.W_K = nn.ParameterList([nn.Parameter(torch.randn(d_model, d_head) * 0.02) for _ in range(n_heads)])
        self.W_V = nn.ParameterList([nn.Parameter(torch.randn(d_model, d_head) * 0.02) for _ in range(n_heads)])
        self.W_O = nn.ParameterList([nn.Parameter(torch.randn(d_head, d_model) * 0.02) for _ in range(n_heads)])
        
        # Causal mask: prevent attending to future tokens
        self.register_buffer("mask", torch.triu(torch.ones(ctx_len, ctx_len), diagonal=1).bool())
    
    def forward(self, x):
        batch, seq_len, _ = x.shape
        attn_out = torch.zeros_like(x)       # accumulated output
        patterns = []                          # store attention patterns for inspection
        
        for h in range(self.n_heads):
            # Project to queries, keys, values
            Q = x @ self.W_Q[h]               # (batch, seq, d_head)
            K = x @ self.W_K[h]               # (batch, seq, d_head)
            V = x @ self.W_V[h]               # (batch, seq, d_head)
            
            # Scaled dot-product attention
            scores = Q @ K.transpose(-2, -1) / (self.d_head ** 0.5)  # (batch, seq, seq)
            scores = scores.masked_fill(self.mask[:seq_len, :seq_len], float("-inf"))
            pattern = F.softmax(scores, dim=-1)                       # (batch, seq, seq)
            patterns.append(pattern)
            
            # Apply attention and project back to residual stream
            head_out = pattern @ V             # (batch, seq, d_head)
            attn_out = attn_out + head_out @ self.W_O[h]  # (batch, seq, d_model)
        
        return attn_out, patterns


class MLP(nn.Module):
    """Simple GELU MLP: d_model -> 4*d_model -> d_model."""
    def __init__(self, d_model):
        super().__init__()
        self.W_in = nn.Linear(d_model, 4 * d_model)
        self.W_out = nn.Linear(4 * d_model, d_model)
    
    def forward(self, x):
        return self.W_out(F.gelu(self.W_in(x)))


class TransformerBlock(nn.Module):
    """Pre-norm transformer block: LN -> Attn -> residual -> LN -> MLP -> residual."""
    def __init__(self, d_model, n_heads, d_head, ctx_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = Attention(d_model, n_heads, d_head, ctx_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model)
    
    def forward(self, x):
        # Attention sublayer with residual
        attn_out, patterns = self.attn(self.ln1(x))
        x_mid = x + attn_out
        
        # MLP sublayer with residual
        mlp_out = self.mlp(self.ln2(x_mid))
        x_out = x_mid + mlp_out
        
        return x_out, attn_out, mlp_out, patterns


class TinyTransformer(nn.Module):
    """A tiny 2-layer transformer that caches ALL intermediate activations.
    
    forward() returns (logits, cache) where cache is a dict with:
      embed, pos_embed, resid_pre_0, attn_out_0, attn_pattern_0, mlp_out_0,
      resid_post_0, resid_pre_1, attn_out_1, attn_pattern_1, mlp_out_1,
      resid_post_1
    """
    def __init__(self, vocab_size, d_model, n_heads, d_head, n_layers, ctx_len):
        super().__init__()
        self.d_model = d_model
        
        # Token and positional embeddings
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(ctx_len, d_model)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_head, ctx_len)
            for _ in range(n_layers)
        ])
        
        self.ln_final = nn.LayerNorm(d_model)
        
        # Unembedding: tied with embedding weights (W_U = W_E^T)
        # This means unembed(x) = x @ W_E^T
    
    def forward(self, tokens):
        batch, seq_len = tokens.shape
        cache = {}
        
        # Embeddings
        tok_embed = self.embed(tokens)                                    # (batch, seq, d_model)
        positions = torch.arange(seq_len, device=tokens.device)
        pos_embed = self.pos_embed(positions)                             # (seq, d_model)
        cache["embed"] = tok_embed
        cache["pos_embed"] = pos_embed
        
        # Initial residual stream = token embed + positional embed
        resid = tok_embed + pos_embed
        
        # Pass through each block, caching everything
        for i, block in enumerate(self.blocks):
            cache[f"resid_pre_{i}"] = resid
            resid, attn_out, mlp_out, patterns = block(resid)
            cache[f"attn_out_{i}"] = attn_out
            cache[f"attn_pattern_{i}"] = patterns    # list of (batch, seq, seq) per head
            cache[f"mlp_out_{i}"] = mlp_out
            cache[f"resid_post_{i}"] = resid
        
        # Final layernorm + unembed (tied weights)
        resid = self.ln_final(resid)
        logits = resid @ self.embed.weight.T                              # (batch, seq, vocab)
        
        return logits, cache


# Instantiate the model
model = TinyTransformer(VOCAB_SIZE, D_MODEL, N_HEADS, D_HEAD, N_LAYERS, CTX_LEN).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"TinyTransformer: {total_params:,} parameters")
print(f"Architecture: {N_LAYERS} layers, {N_HEADS} heads, d_model={D_MODEL}, d_head={D_HEAD}")

## Part 2: A Toy Task — Sequence Copying

We need a task that's simple enough for our tiny model to learn, but interesting enough to show real interpretability phenomena. Here's the plan:

**Input:** `[A, B, C, SEP, A, B, _]`  
**Target:** Model should predict `C` at the `_` position.

The model sees three tokens, then a separator, then the first two tokens repeated. It must predict the third token — the one that's "missing" from the repeated sequence.

This is a simplified version of the kind of **lookup/copying behavior** that shows up in the Indirect Object Identification (IOI) task studied by Wang et al. (2022). The model needs to:
1. Identify which position to copy from (the token after B, before SEP)
2. Actually copy that token's identity to the output

These two steps map cleanly onto attention heads with different roles

In [ ]:
# === Data generation for the sequence copying task ===

SEP_TOKEN = 0  # Token 0 is the separator; tokens 1..19 are "regular" tokens

def generate_batch(batch_size):
    """Generate a batch of copying-task sequences.
    
    Format: [A, B, C, SEP, A, B, PAD] where PAD position should predict C.
    A, B, C are distinct random tokens from 1..19.
    """
    seqs = torch.zeros(batch_size, 7, dtype=torch.long)
    targets = torch.zeros(batch_size, dtype=torch.long)
    
    for i in range(batch_size):
        # Pick 3 distinct tokens for A, B, C
        tokens = torch.randperm(VOCAB_SIZE - 1)[:3] + 1  # avoid 0 (SEP)
        A, B, C = tokens[0].item(), tokens[1].item(), tokens[2].item()
        
        seqs[i] = torch.tensor([A, B, C, SEP_TOKEN, A, B, C])  # last C is input context
        targets[i] = C  # model must predict C at position 6
    
    # We feed positions 0..5 as input and train the model to predict position 6
    # But for causal LM, we feed all 7 tokens and take loss at position 6
    return seqs, targets


# === Training loop ===
torch.manual_seed(42)
model = TinyTransformer(VOCAB_SIZE, D_MODEL, N_HEADS, D_HEAD, N_LAYERS, CTX_LEN).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training on sequence copying task...")
print("Task: [A, B, C, SEP, A, B, ?] -> predict C\n")

for step in range(501):
    # Generate a fresh batch each step
    seqs, targets = generate_batch(256)
    seqs, targets = seqs.to(device), targets.to(device)
    
    # Forward pass: get logits at position 5 (predicting token at position 6)
    # We feed the first 6 tokens (positions 0..5) and predict position 6
    logits, _ = model(seqs[:, :6])  # feed [A, B, C, SEP, A, B]
    pred_logits = logits[:, -1, :]  # logits at the last input position
    
    loss = F.cross_entropy(pred_logits, targets)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if step % 100 == 0:
        acc = (pred_logits.argmax(-1) == targets).float().mean().item()
        print(f"Step {step:4d} | Loss: {loss.item():.4f} | Accuracy: {acc:.1%}")

# === Test evaluation ===
model.eval()
with torch.no_grad():
    test_seqs, test_targets = generate_batch(1000)
    test_logits, _ = model(test_seqs[:, :6])
    test_preds = test_logits[:, -1, :].argmax(-1)
    test_acc = (test_preds == test_targets).float().mean().item()
    print(f"\nTest accuracy (1000 examples): {test_acc:.1%}")

## Part 3: Looking Inside — The Residual Stream

The **residual stream** is the central highway of a transformer. At each layer, the attention output and MLP output are *added* to the residual stream:

```
resid_post_0 = resid_pre_0 + attn_out_0 + mlp_out_0
resid_post_1 = resid_pre_1 + attn_out_1 + mlp_out_1
```

This additive structure is what makes transformers so interpretable — every component writes *additively* to the residual stream, and the final output is just the sum of all contributions.

Let's verify this is actually true in our model, then visualize how much each component contributes.

In [ ]:
# === Verify the residual stream structure ===

# Generate a single test example
torch.manual_seed(123)
test_seq, test_target = generate_batch(1)
test_input = test_seq[:, :6].to(device)

with torch.no_grad():
    logits, cache = model(test_input)

# Verify: resid_post = resid_pre + attn_out + mlp_out (for each layer)
for layer in range(N_LAYERS):
    resid_pre = cache[f"resid_pre_{layer}"]
    attn_out = cache[f"attn_out_{layer}"]
    mlp_out = cache[f"mlp_out_{layer}"]
    resid_post = cache[f"resid_post_{layer}"]
    
    # The residual stream is additive: pre + attn + mlp = post
    # Note: our block does LN->Attn, then adds to resid, then LN->MLP, adds to resid
    # So resid_post = resid_pre + attn_out + mlp_out
    reconstructed = resid_pre + attn_out + mlp_out
    max_diff = (resid_post - reconstructed).abs().max().item()
    print(f"Layer {layer}: max|resid_post - (resid_pre + attn_out + mlp_out)| = {max_diff:.2e}")

# === Visualize component norms at the final position ===
pos = -1  # last position (where the prediction happens)

component_names = []
component_norms = []

# Embedding contribution
embed_norm = (cache["embed"][0, pos] + cache["pos_embed"][pos]).norm().item()
component_names.append("embed+pos")
component_norms.append(embed_norm)

# Each layer's attention and MLP contributions
for layer in range(N_LAYERS):
    attn_norm = cache[f"attn_out_{layer}"][0, pos].norm().item()
    mlp_norm = cache[f"mlp_out_{layer}"][0, pos].norm().item()
    component_names.append(f"attn_{layer}")
    component_norms.append(attn_norm)
    component_names.append(f"mlp_{layer}")
    component_norms.append(mlp_norm)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
ax.bar(component_names, component_norms, color=colors)
ax.set_ylabel("L2 Norm")
ax.set_title("Residual Stream Component Norms (final position)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Attention Patterns — What Is the Model Looking At?

Attention patterns tell us **what information flows where**. Each attention head produces a matrix of shape `(seq_len, seq_len)` where entry `[i, j]` says "how much does position `i` attend to position `j`."

For our copying task (`[A, B, C, SEP, A, B, ?]` -> predict `C`), we'd expect to see:
- Some head that attends from the final position back to position 2 (where `C` lives)
- Possibly a head that attends to the SEP token or uses positional information to figure out which token to copy

Let's look at all 4 heads (2 layers x 2 heads).

In [ ]:
# === Visualize attention patterns for all heads ===

# Get a test example with known tokens
torch.manual_seed(77)
test_seq, test_target = generate_batch(1)
test_input = test_seq[:, :6].to(device)
token_ids = test_input[0].tolist()

# Create readable labels
label_map = {SEP_TOKEN: "SEP"}
token_labels = []
for idx, t in enumerate(token_ids):
    if t == SEP_TOKEN:
        token_labels.append("SEP")
    else:
        token_labels.append(f"{t}")

# Annotate with position info
pos_labels = [f"p{i}:{token_labels[i]}" for i in range(len(token_labels))]
print(f"Sequence: {pos_labels}")
print(f"Target: {test_target.item()} (token at position 2)")

with torch.no_grad():
    logits, cache = model(test_input)
    pred = logits[0, -1].argmax().item()
    print(f"Prediction: {pred} ({'correct' if pred == test_target.item() else 'wrong'})\n")

# Plot 2x2 grid: rows=layers, cols=heads
fig, axes = plt.subplots(N_LAYERS, N_HEADS, figsize=(12, 10))

for layer in range(N_LAYERS):
    patterns = cache[f"attn_pattern_{layer}"]  # list of (1, seq, seq) per head
    for head in range(N_HEADS):
        ax = axes[layer, head]
        pattern = patterns[head][0].cpu().numpy()  # (seq, seq)
        
        im = ax.imshow(pattern, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(pos_labels)))
        ax.set_yticks(range(len(pos_labels)))
        ax.set_xticklabels(pos_labels, rotation=45, ha="right", fontsize=9)
        ax.set_yticklabels(pos_labels, fontsize=9)
        ax.set_xlabel("Source (attending to)")
        ax.set_ylabel("Destination (attending from)")
        ax.set_title(f"Layer {layer}, Head {head}")
        plt.colorbar(im, ax=ax, fraction=0.046)
        
        # Print what the final position attends to
        final_attn = pattern[-1]
        top_pos = final_attn.argmax()
        print(f"L{layer}H{head}: final pos attends most to position {top_pos} "
              f"(token={token_labels[top_pos]}, weight={final_attn[top_pos]:.2f})")

plt.suptitle("Attention Patterns — What Is Each Head Looking At?", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 5: The Logit Lens — From Scratch

The **logit lens** (introduced by nostalgebraist) is a beautifully simple idea: at each layer, take the current residual stream and project it through the unembedding matrix to see what the model would predict *if it stopped processing right now*.

If the model's residual stream at layer 0 already "knows" the answer, we'll see the correct token ranked highly early on. If it only figures it out in layer 1, we'll see the probability jump.

Since we tied our embedding and unembedding weights, the unembedding operation is just:

```python
logits = residual_stream @ W_E.T
```

**Note:** Our implementation below applies the final LayerNorm before unembedding, which means it is closer to the **tuned lens** approach than the original logit lens by nostalgebraist (which skips the final LayerNorm). This tends to give cleaner results but is technically a different method.

Let's implement this ourselves.

In [ ]:
# === Logit Lens: project intermediate residual streams to token probabilities ===

torch.manual_seed(77)
test_seq, test_target = generate_batch(1)
test_input = test_seq[:, :6].to(device)
target_token = test_target.item()

with torch.no_grad():
    logits, cache = model(test_input)

# Unembedding matrix (W_E^T since we use tied weights)
W_U = model.embed.weight.T  # (d_model, vocab_size)

# Collect residual streams at each stage
stages = []
stage_names = []

# After embedding
stages.append(cache["embed"] + cache["pos_embed"])
stage_names.append("embed")

# After each sublayer
for layer in range(N_LAYERS):
    stages.append(cache[f"resid_pre_{layer}"] + cache[f"attn_out_{layer}"])
    stage_names.append(f"L{layer}_attn")
    stages.append(cache[f"resid_post_{layer}"])
    stage_names.append(f"L{layer}_mlp")

print(f"Target token: {target_token}")
print(f"\nLogit Lens — Top-3 predictions at final position:\n")

target_probs_by_stage = []

for name, resid in zip(stage_names, stages):
    # Apply layer norm then unembed (matching the model's actual computation)
    normed = model.ln_final(resid)
    stage_logits = normed @ W_U   # (1, seq, vocab)
    probs = F.softmax(stage_logits[0, -1], dim=-1)  # final position
    
    # Top 3 predictions
    top3_probs, top3_ids = probs.topk(3)
    top3_str = ", ".join([f"{top3_ids[j].item()}({top3_probs[j].item():.2f})" for j in range(3)])
    target_prob = probs[target_token].item()
    target_probs_by_stage.append(target_prob)
    
    marker = " <-- correct!" if top3_ids[0].item() == target_token else ""
    print(f"  {name:10s}: top3=[{top3_str}]  P(target={target_token})={target_prob:.3f}{marker}")

# === Heatmap: target token probability at each layer and position ===
seq_len = test_input.shape[1]
prob_grid = np.zeros((len(stages), seq_len))

for i, resid in enumerate(stages):
    normed = model.ln_final(resid)
    stage_logits = normed @ W_U
    probs = F.softmax(stage_logits[0], dim=-1)  # (seq, vocab)
    prob_grid[i, :] = probs[:, target_token].cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(prob_grid, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)

token_labels = [f"p{i}:{test_input[0, i].item()}" for i in range(seq_len)]
ax.set_xticks(range(seq_len))
ax.set_xticklabels(token_labels, rotation=45, ha="right")
ax.set_yticks(range(len(stage_names)))
ax.set_yticklabels(stage_names)
ax.set_xlabel("Position")
ax.set_ylabel("Layer")
ax.set_title(f"Logit Lens: P(target={target_token}) at each layer and position")
plt.colorbar(im, ax=ax, label="Probability")

# Annotate cells
for i in range(len(stages)):
    for j in range(seq_len):
        val = prob_grid[i, j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

plt.tight_layout()
plt.show()

## Part 6: Activation Patching — From Scratch

**Activation patching** (also called causal tracing) is the gold standard for figuring out which components matter. The protocol:

1. **Clean run:** Feed the correct input `[A, B, C, SEP, A, B]`. The model predicts `C`. Record all activations.
2. **Corrupted run:** Feed a modified input `[A, B, D, SEP, A, B]` (swap `C` for `D`). The model now predicts `D`.
3. **Patch:** Take the corrupted run, but **replace one component's activation** with the clean version. If the model's answer flips back toward `C`, that component was carrying the critical information.

We measure recovery as: how much does the logit difference (logit_C - logit_D) shift toward the clean value when we patch?

In [ ]:
# === Activation Patching ===

def run_with_patch(model, tokens, patch_component, patch_value):
    """Run the model but replace one component's output with a patched value.
    
    This is a manual forward pass that lets us intervene on specific components.
    """
    batch, seq_len = tokens.shape
    
    # Embedding
    tok_embed = model.embed(tokens)
    positions = torch.arange(seq_len, device=tokens.device)
    pos_embed = model.pos_embed(positions)
    resid = tok_embed + pos_embed
    
    for i, block in enumerate(model.blocks):
        # Attention
        attn_out, _ = block.attn(block.ln1(resid))
        if patch_component == f"attn_out_{i}":
            attn_out = patch_value  # PATCH: replace with clean activation
        resid = resid + attn_out
        
        # MLP
        mlp_out = block.mlp(block.ln2(resid))
        if patch_component == f"mlp_out_{i}":
            mlp_out = patch_value   # PATCH: replace with clean activation
        resid = resid + mlp_out
    
    resid = model.ln_final(resid)
    logits = resid @ model.embed.weight.T
    return logits


# Generate clean and corrupted inputs
torch.manual_seed(42)
# Clean: [A, B, C, SEP, A, B]
A, B, C, D = 3, 7, 12, 15  # specific token choices
clean_input = torch.tensor([[A, B, C, SEP_TOKEN, A, B]], device=device)
corrupted_input = torch.tensor([[A, B, D, SEP_TOKEN, A, B]], device=device)

print(f"Clean input:     [A={A}, B={B}, C={C}, SEP, A={A}, B={B}] -> should predict {C}")
print(f"Corrupted input: [A={A}, B={B}, D={D}, SEP, A={A}, B={B}] -> should predict {D}\n")

# Get clean and corrupted activations
with torch.no_grad():
    clean_logits, clean_cache = model(clean_input)
    corrupted_logits, corrupted_cache = model(corrupted_input)

# Baseline logit differences at the final position
clean_logit_diff = (clean_logits[0, -1, C] - clean_logits[0, -1, D]).item()
corrupted_logit_diff = (corrupted_logits[0, -1, C] - corrupted_logits[0, -1, D]).item()

print(f"Clean logit diff    (logit_C - logit_D): {clean_logit_diff:+.2f}")
print(f"Corrupted logit diff (logit_C - logit_D): {corrupted_logit_diff:+.2f}")
print(f"Gap to recover: {clean_logit_diff - corrupted_logit_diff:.2f}\n")

# Patch each component and measure recovery
components = ["attn_out_0", "mlp_out_0", "attn_out_1", "mlp_out_1"]
recoveries = []

for comp in components:
    with torch.no_grad():
        patched_logits = run_with_patch(
            model, corrupted_input, 
            patch_component=comp, 
            patch_value=clean_cache[comp]
        )
    patched_diff = (patched_logits[0, -1, C] - patched_logits[0, -1, D]).item()
    
    # Recovery = how much of the gap did we close?
    total_gap = clean_logit_diff - corrupted_logit_diff
    recovery = (patched_diff - corrupted_logit_diff) / total_gap if abs(total_gap) > 1e-6 else 0.0
    recoveries.append(recovery)
    print(f"Patch {comp:12s}: logit_diff={patched_diff:+.2f}, recovery={recovery:.1%}")

# === Bar chart ===
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#DD8452" if "attn" in c else "#55A868" for c in components]
ax.bar(components, recoveries, color=colors)
ax.set_ylabel("Fraction of Logit Diff Recovered")
ax.set_title("Activation Patching: Which Components Carry the Answer?")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.axhline(y=1, color="gray", linestyle="--", linewidth=0.5, label="Full recovery")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Annotate bars with percentage values
for i, (comp, rec) in enumerate(zip(components, recoveries)):
    ax.text(i, rec + 0.02, f"{rec:.0%}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

## Part 7: The OV Circuit — What Does a Head Copy?

Each attention head can be decomposed into two circuits:
- **QK circuit** (W_Q^T W_K): determines *where* the head attends (source-destination matching)
- **OV circuit** (W_V W_O): determines *what* gets written to the residual stream when the head attends to a token

The full OV circuit for a head, from input token to output logit, is:

```
W_E @ W_V @ W_O @ W_U
```

**Note:** This formula omits LayerNorm, which is standard practice for OV circuit analysis — LayerNorm approximately scales all directions equally, so the relative structure of the OV matrix is preserved.

This matrix tells us: "if this head attends to input token X, what does it write to the output logits?" A **copying head** will have a strong diagonal — token X in, token X out. Let's check.

In [ ]:
# === OV Circuit Analysis ===

# Get the embedding and unembedding matrices
W_E = model.embed.weight  # (vocab, d_model)
W_U = model.embed.weight.T  # (d_model, vocab) — tied weights

fig, axes = plt.subplots(N_LAYERS, N_HEADS, figsize=(12, 10))

for layer in range(N_LAYERS):
    attn = model.blocks[layer].attn
    for head in range(N_HEADS):
        ax = axes[layer, head]
        
        # Full OV circuit: token_in -> embed -> W_V -> W_O -> unembed -> logit
        # Shape: (vocab, d_model) @ (d_model, d_head) @ (d_head, d_model) @ (d_model, vocab)
        with torch.no_grad():
            W_V = attn.W_V[head]   # (d_model, d_head)
            W_O = attn.W_O[head]   # (d_head, d_model)
            
            OV_circuit = W_E @ W_V @ W_O @ W_U  # (vocab, vocab)
        
        ov = OV_circuit.cpu().numpy()
        
        # Measure how "copy-like" this head is
        # A perfect copy head has the identity matrix
        diag_mean = np.mean(np.diag(ov))
        offdiag_mean = (np.sum(ov) - np.trace(ov)) / (VOCAB_SIZE * (VOCAB_SIZE - 1))
        
        im = ax.imshow(ov, cmap="RdBu_r", 
                       vmin=-np.abs(ov).max(), vmax=np.abs(ov).max())
        ax.set_xlabel("Output token")
        ax.set_ylabel("Input token")
        ax.set_title(f"L{layer}H{head} OV Circuit\ndiag_mean={diag_mean:.2f}")
        plt.colorbar(im, ax=ax, fraction=0.046)
        
        # Check if diagonal is dominant (copy behavior)
        is_copy = diag_mean > 0.5
        if is_copy:
            print(f"L{layer}H{head}: COPYING HEAD (diagonal mean={diag_mean:.2f})")
        else:
            print(f"L{layer}H{head}: Not a strong copy head (diagonal mean={diag_mean:.2f})")

plt.suptitle("OV Circuits: What Does Each Head Write?", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nA strong diagonal means: 'if I attend to token X, I boost token X in the output.'")
print("This is the signature of a copying head.")

## What You Just Built

Congratulations — you just implemented, from scratch, the core toolkit of mechanistic interpretability:

1. **A transformer** with cached activations (what TransformerLens calls `run_with_cache`)
2. **The residual stream view** — verifying that every component writes additively
3. **Attention pattern visualization** — seeing where information flows
4. **The logit lens** — reading off intermediate predictions from the residual stream
5. **Activation patching** — causally testing which components matter
6. **OV circuit analysis** — understanding what a head copies

These are the same techniques that TransformerLens automates. The rest of this guide uses TransformerLens for convenience, but now you know **exactly** what's happening under the hood.

**Key takeaways:**
- The residual stream is a sum of additive contributions — this is what makes transformers interpretable
- Attention patterns tell you *where* information moves; OV circuits tell you *what* gets moved
- Activation patching is the gold standard for causal claims about model internals
- The logit lens lets you watch the model "make up its mind" layer by layer

Next up: we'll use TransformerLens to do all of this (and more) on real language models.